# Data Integration

## Business Objective

The objective of this notebook is to integrate the cleaned e-commerce datasets into a reliable analytical data model suitable for downstream business analysis, SQL querying, dashboard development, and executive reporting.

Rather than analyzing each table independently, this phase connects the cleaned datasets through their relational keys to create a consistent view of customers, orders, products, reviews, and user behavior.

This notebook focuses on:

- Loading cleaned datasets
- Understanding table relationships
- Validating primary and foreign keys
- Building an order-level analytical dataset
- Building an item-level sales dataset
- Creating supporting behavioral and review datasets
- Exporting integrated datasets for EDA, SQL, and Power BI

## Integration Methodology

The integration process follows a structured workflow:

1. Load cleaned datasets  
2. Inspect dataset dimensions  
3. Define relational schema  
4. Validate primary keys  
5. Validate foreign key relationships  
6. Merge transactional tables  
7. Validate row counts after joins  
8. Create analysis-ready datasets  
9. Export integrated outputs  

The goal is to ensure that all downstream analysis is based on a trustworthy and well-documented analytical data model.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Set Project Directory

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "cleaned"
INTEGRATED_DATA_DIR = PROJECT_ROOT / "data" / "integrated"

INTEGRATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis')

## Load Cleaned Datasets

In [3]:
users = pd.read_csv(CLEAN_DATA_DIR / "users_clean.csv")
products = pd.read_csv(CLEAN_DATA_DIR / "products_clean.csv")
orders = pd.read_csv(CLEAN_DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(CLEAN_DATA_DIR / "order_items_clean.csv")
reviews = pd.read_csv(CLEAN_DATA_DIR / "reviews_clean.csv")
events = pd.read_csv(CLEAN_DATA_DIR / "events_clean.csv")

## Convert Date Columns

In [4]:
users["signup_date"] = pd.to_datetime(users["signup_date"], errors="coerce")
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
reviews["review_date"] = pd.to_datetime(reviews["review_date"], errors="coerce")
events["event_timestamp"] = pd.to_datetime(events["event_timestamp"], errors="coerce")

# Dataset Overview

Before integrating the datasets, the dimensions of each cleaned table are reviewed to confirm that the cleaned files were loaded correctly.

In [5]:
datasets = {
    "users": users,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
    "events": events
}

overview = pd.DataFrame({
    "table": list(datasets.keys()),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()]
})

overview

,table,rows,columns
0,users,10000,6
1,products,2000,6
2,orders,20000,5
3,order_items,43525,7
4,reviews,15000,7
5,events,80000,5


## Interpretation

The cleaned datasets have been successfully loaded and are ready for relational validation. At this stage, no joins have been performed yet. The next step is to define how the tables should connect through their primary and foreign keys.

# Relational Schema

The e-commerce database follows a relational structure.

## Core Relationships

- `users.user_id` connects to `orders.user_id`
- `orders.order_id` connects to `order_items.order_id`
- `products.product_id` connects to `order_items.product_id`
- `users.user_id` connects to `reviews.user_id`
- `products.product_id` connects to `reviews.product_id`
- `users.user_id` connects to `events.user_id`
- `products.product_id` connects to `events.product_id`

## Analytical Model

For business analysis, two primary datasets will be created:

1. **Item-level sales dataset**  
   Combines orders, order items, products, and users.  
   This dataset supports revenue, product, category, customer, and geographic analysis.

2. **Behavioral events dataset**  
   Combines events, users, and products.  
   This dataset supports funnel analysis and customer behavior analysis.

# Primary Key Validation

Before joining tables, each primary key is validated to ensure uniqueness. Duplicate primary keys would create unreliable joins and may inflate row counts during integration.

In [6]:
primary_key_checks = pd.DataFrame({
    "table": ["users", "products", "orders", "order_items", "reviews", "events"],
    "primary_key": ["user_id", "product_id", "order_id", "order_item_id", "review_id", "event_id"],
    "rows": [
        len(users),
        len(products),
        len(orders),
        len(order_items),
        len(reviews),
        len(events)
    ],
    "unique_primary_keys": [
        users["user_id"].nunique(),
        products["product_id"].nunique(),
        orders["order_id"].nunique(),
        order_items["order_item_id"].nunique(),
        reviews["review_id"].nunique(),
        events["event_id"].nunique()
    ],
    "duplicate_primary_keys": [
        users["user_id"].duplicated().sum(),
        products["product_id"].duplicated().sum(),
        orders["order_id"].duplicated().sum(),
        order_items["order_item_id"].duplicated().sum(),
        reviews["review_id"].duplicated().sum(),
        events["event_id"].duplicated().sum()
    ]
})

primary_key_checks

,table,primary_key,rows,unique_primary_keys,duplicate_primary_keys
0,users,user_id,10000,10000,0
1,products,product_id,2000,2000,0
2,orders,order_id,20000,20000,0
3,order_items,order_item_id,43525,43525,0
4,reviews,review_id,15000,15000,0
5,events,event_id,80000,80000,0


## Primary Key Interpretation

The primary key validation confirms whether each table can be safely used in relational joins. A duplicate primary key could cause duplicated rows after merging and distort revenue, customer, product, or event metrics.

If all duplicate primary key counts equal zero, the tables are structurally ready for integration.

# Foreign Key Validation

Foreign key validation checks whether records in one table correctly reference records in another table.

This step is important because invalid foreign keys can cause missing values after joins and may indicate broken relationships between tables.

In [7]:
foreign_key_checks = pd.DataFrame({
    "relationship": [
        "orders.user_id → users.user_id",
        "order_items.order_id → orders.order_id",
        "order_items.product_id → products.product_id",
        "reviews.user_id → users.user_id",
        "reviews.product_id → products.product_id",
        "events.user_id → users.user_id",
        "events.product_id → products.product_id"
    ],
    "foreign_key_rows": [
        orders["user_id"].notna().sum(),
        order_items["order_id"].notna().sum(),
        order_items["product_id"].notna().sum(),
        reviews["user_id"].notna().sum(),
        reviews["product_id"].notna().sum(),
        events["user_id"].notna().sum(),
        events["product_id"].notna().sum()
    ],
    "unmatched_foreign_keys": [
        (~orders["user_id"].isin(users["user_id"])).sum(),
        (~order_items["order_id"].isin(orders["order_id"])).sum(),
        (~order_items["product_id"].isin(products["product_id"])).sum(),
        (~reviews["user_id"].isin(users["user_id"])).sum(),
        (~reviews["product_id"].isin(products["product_id"])).sum(),
        (~events["user_id"].isin(users["user_id"])).sum(),
        (~events["product_id"].isin(products["product_id"])).sum()
    ]
})

foreign_key_checks

,relationship,foreign_key_rows,unmatched_foreign_keys
0,orders.user_id → users.user_id,20000,0
1,order_items.order_id → orders.order_id,43525,0
2,order_items.product_id → products.product_id,43525,0
3,reviews.user_id → users.user_id,15000,0
4,reviews.product_id → products.product_id,15000,0
5,events.user_id → users.user_id,80000,0
6,events.product_id → products.product_id,77643,2357


## Foreign Key Interpretation

Foreign key validation helps determine whether relationships between tables are reliable. If unmatched foreign keys exist, they should be documented and considered when interpreting joined datasets.

For portfolio purposes, documenting unmatched relationships is important because it demonstrates that joins were not performed blindly.

# Build Item-Level Sales Dataset

The item-level sales dataset combines:

- Order item information
- Order-level transaction details
- Product details
- Customer details

This dataset will support most business analytics use cases, including revenue analysis, product performance, category analysis, customer segmentation, and geographic sales analysis.

## Step 1: Merge Order Items with Orders

In [8]:
sales_items = order_items.merge(
    orders,
    on=["order_id", "user_id"],
    how="left",
    indicator=True
)

sales_items["_merge"].value_counts()

_merge
both          43525
left_only         0
right_only        0
Name: count, dtype: int64

## Validate Order Join

In [9]:
unmatched_orders = sales_items[sales_items["_merge"] != "both"]

unmatched_orders.shape[0]

0

## Remove Join Indicator

In [10]:
sales_items = sales_items.drop(columns="_merge")

## Step 2: Merge Sales Items with Products

In [11]:
sales_items = sales_items.merge(
    products,
    on="product_id",
    how="left",
    indicator=True
)

sales_items["_merge"].value_counts()

_merge
both          43525
left_only         0
right_only        0
Name: count, dtype: int64

## Validate Product Join

In [12]:
unmatched_products = sales_items[sales_items["_merge"] != "both"]

unmatched_products.shape[0]

0

## Remove Join Indicator

In [13]:
sales_items = sales_items.drop(columns="_merge")

## Step 3: Merge Sales Items with Users

In [14]:
sales_items = sales_items.merge(
    users,
    on="user_id",
    how="left",
    indicator=True
)

sales_items["_merge"].value_counts()

_merge
both          43525
left_only         0
right_only        0
Name: count, dtype: int64

## Validate User Join

In [15]:
unmatched_users = sales_items[sales_items["_merge"] != "both"]

unmatched_users.shape[0]

0

## Remove Join Indicator

In [16]:
sales_items = sales_items.drop(columns="_merge")

## Create Revenue Column

Although order-level totals are available, item-level revenue is calculated from quantity and item price to support product and category-level analysis.

In [17]:
sales_items["item_revenue"] = sales_items["quantity"] * sales_items["item_price"]

sales_items[["quantity", "item_price", "item_revenue"]].head()

,quantity,item_price,item_revenue
0,2.0,8.07,16.14
1,1.0,68.43,68.43
2,1.0,114.66,114.66
3,1.0,124.68,124.68
4,1.0,40.71,40.71


## Sales Dataset Preview

In [18]:
sales_items.head()

,order_item_id,order_id,product_id,user_id,quantity,item_price,item_total,order_date,order_status,total_amount,...,category,brand,price,rating,name,email,gender,city,signup_date,item_revenue
0,I00000001,O00000001,P001758,U009310,2.0,8.07,16.14,2025-09-09 14:52:37.292731,Processing,689.66,...,Pet Supplies,Everest,8.07,2.69,Christine Snyder,john14@example.com,Female,Jonestown,2024-02-22,16.14
1,I00028796,O00013244,P000446,U009679,1.0,68.43,68.43,2024-01-12 14:25:12.736023,Cancelled,68.43,...,Clothing,GreenLeaf,68.43,3.24,Matthew Jordan,losborne@example.com,Other,Gregorymouth,2024-02-01,68.43
2,I00028798,O00013245,P001420,U007391,1.0,114.66,114.66,2025-08-26 21:26:34.877799,Completed,NaN,...,Clothing,NeoTech,114.66,2.61,Linda Gonzalez,gary18@example.com,Male,NaN,2024-10-25,114.66
3,I00028799,O00013246,P001955,U004554,1.0,124.68,124.68,2025-05-18 00:37:20.715594,Returned,124.68,...,Beauty,Astra,124.68,4.88,Kristen Villarreal,anneschultz@example.com,Other,West Donaldshire,2025-05-29,124.68
4,I00028800,O00013247,P001472,U006926,1.0,40.71,40.71,2024-04-24 09:30:51.674828,Shipped,40.71,...,Toys,Zenith,40.71,3.53,Ann Martin,pam28@example.com,Other,Port Amandaside,2024-11-18,40.71


## Sales Dataset Shape

In [19]:
sales_items.shape

(43525, 21)

## Sales Dataset Interpretation

The item-level sales dataset is the primary analytical dataset for revenue and product performance analysis. Each row represents a purchased item within an order, enriched with order, product, and customer attributes.

This dataset will be used heavily in EDA, SQL analysis, and dashboard development.

# Build Reviews Analytical Dataset

The reviews dataset is enriched with product and customer attributes to support customer satisfaction and product feedback analysis.

In [20]:
reviews_analysis = reviews.merge(
    users,
    on="user_id",
    how="left"
).merge(
    products,
    on="product_id",
    how="left"
)

reviews_analysis.shape

(15000, 17)

## Reviews Dataset Preview

In [21]:
reviews_analysis.head()

,review_id,order_id,product_id,user_id,rating_x,review_text,review_date,name,email,gender,city,signup_date,product_name,category,brand,price,rating_y
0,R00000528,O00000237,P001326,U001094,2.0,Color was different from images.,2025-10-14 12:03:56.749446,Lisa Daugherty,hoffmanjared@example.com,Other,North Donald,2024-08-15,GreenLeaf Interview,Clothing,GreenLeaf,61.39,4.00
1,R00018553,O00008568,P001344,U005616,3.0,Value for money.,2024-07-21 14:27:37.749730,Timothy Grant,paula52@example.net,Male,North Kylechester,2024-03-07,Orion Second,Pet Supplies,Orion,103.47,2.72
2,R00040036,O00018362,P001706,U007127,3.0,Highly recommend this brand.,2024-04-29 09:39:16.110742,Jon Mitchell,karen16@example.com,Other,Tiffanyland,2025-07-09,Willow Measure,Electronics,Willow,970.10,4.04
3,R00023707,O00010937,P001852,U008044,1.0,Fast shipping and good packaging.,2024-01-28 22:47:22.836308,Lauren Allen,qgarcia@example.net,NaN,Leemouth,2024-09-08,Everest All,Automotive,Everest,643.14,4.29
4,R00014523,O00006692,P000224,U004164,5.0,"Not as expected, quality is poor.",2024-04-27 02:24:03.789920,Jessica Smith,NaN,Female,Port Brianchester,2024-12-19,Astra Else,Automotive,Astra,88.96,2.52


## Reviews Dataset Interpretation

The integrated reviews dataset connects customer feedback to both customer demographics and product attributes. This enables analysis of customer satisfaction by product category, city, customer segment, and product rating patterns.

# Build Events Analytical Dataset

The events dataset is enriched with customer and product information to support behavioral analysis and conversion funnel analysis.

In [22]:
events_analysis = events.merge(
    users,
    on="user_id",
    how="left"
).merge(
    products,
    on="product_id",
    how="left"
)

events_analysis.shape

(80000, 15)

## Events Dataset Preview

In [23]:
events_analysis.head()

,event_id,user_id,product_id,event_type,event_timestamp,name,email,gender,city,signup_date,product_name,category,brand,price,rating
0,E00000001,U009798,P001393,Cart,2025-07-08 14:28:55.893919,Rebecca Blanchard,teresamullins@example.net,Other,Lake Nicholeshire,2024-01-27,Willow Several,Automotive,Willow,314.95,3.10
1,E00052489,U000716,P000628,Cart,2025-05-27 23:42:22.765378,Paige Weber,zterry@example.net,Other,Roweport,2025-03-16,Astra Small,Electronics,Astra,612.23,4.58
2,E00052487,U002538,P001302,Cart,2024-12-22 03:54:52.939093,Jose Scott,jenniferpeters@example.com,Other,NaN,2025-01-25,Astra Establish,Sports,Astra,13.29,4.64
3,E00052486,U000227,P000979,View,2024-12-03 10:49:29.272881,Mark Walters,whenry@example.net,Female,Reynoldsshire,2025-10-24,NaN,Home & Kitchen,Solace,23.87,3.53
4,E00052485,U000825,P000047,View,2024-06-17 18:30:23.128829,Wayne Rodriguez,jacobbrown@example.org,Male,West Tonya,2024-11-28,Acme Together,Beauty,Acme,94.71,3.71


## Events Dataset Interpretation

The integrated events dataset connects user behavior to customer and product attributes. This dataset can support conversion funnel analysis, product engagement analysis, and user journey exploration.

# Data Model Validation

This section validates the integrated datasets by checking row counts, missing values, and duplicate records after joins.

In [24]:
integrated_validation = pd.DataFrame({
    "dataset": ["sales_items", "reviews_analysis", "events_analysis"],
    "rows": [
        len(sales_items),
        len(reviews_analysis),
        len(events_analysis)
    ],
    "columns": [
        sales_items.shape[1],
        reviews_analysis.shape[1],
        events_analysis.shape[1]
    ],
    "duplicate_rows": [
        sales_items.duplicated().sum(),
        reviews_analysis.duplicated().sum(),
        events_analysis.duplicated().sum()
    ],
    "missing_values_total": [
        sales_items.isna().sum().sum(),
        reviews_analysis.isna().sum().sum(),
        events_analysis.isna().sum().sum()
    ]
})

integrated_validation

,dataset,rows,columns,duplicate_rows,missing_values_total
0,sales_items,43525,21,0,18022
1,reviews_analysis,15000,17,0,4883
2,events_analysis,80000,15,0,35035


## Validation Interpretation

The validation summary confirms that integrated datasets were created successfully. Remaining missing values may be expected due to missing values preserved from the cleaning phase or unmatched relationships identified during foreign key validation.

These integrated datasets are suitable for exploratory analysis because major structural issues such as duplicate primary keys and exact duplicate records were already addressed during cleaning.

# Export Integrated Datasets

The integrated datasets are exported to the `data/integrated` directory for use in EDA, SQL analysis, Power BI, and reporting.

In [25]:
sales_items.to_csv(INTEGRATED_DATA_DIR / "sales_items_analysis.csv", index=False)
reviews_analysis.to_csv(INTEGRATED_DATA_DIR / "reviews_analysis.csv", index=False)
events_analysis.to_csv(INTEGRATED_DATA_DIR / "events_analysis.csv", index=False)

list(INTEGRATED_DATA_DIR.iterdir())

[PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/integrated/sales_items_analysis.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/integrated/events_analysis.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/data/integrated/reviews_analysis.csv')]

# Executive Summary

## Objective

The objective of this notebook was to integrate the cleaned e-commerce datasets into analysis-ready datasets suitable for business analysis and visualization.

## Integration Work Completed

The following work was completed:

- Loaded all cleaned datasets from the `data/cleaned` directory.
- Validated primary key uniqueness across all core tables.
- Validated foreign key relationships between related tables.
- Created an item-level sales dataset by joining order items, orders, products, and users.
- Created an enriched reviews dataset by joining reviews with users and products.
- Created an enriched events dataset by joining events with users and products.
- Created an item-level revenue field to support product and category-level sales analysis.
- Validated integrated datasets after joins.
- Exported integrated analytical datasets to the `data/integrated` directory.

## Final Analytical Outputs

The following integrated datasets were created:

- `sales_items_analysis.csv`
- `reviews_analysis.csv`
- `events_analysis.csv`

## Business Value

The integrated data model transforms separate cleaned CSV files into structured analytical datasets. This allows the project to move from data preparation into business insight generation.

The item-level sales dataset supports revenue, product, customer, category, and geographic analysis. The reviews dataset supports customer satisfaction analysis. The events dataset supports customer behavior and funnel analysis.

## Next Steps

The next stage of the project is Exploratory Data Analysis (EDA). In that phase, the integrated datasets will be used to answer business questions such as:

- Which categories generate the most revenue?
- Which products are top performers?
- How does revenue change over time?
- Which cities contain the most customers?
- How do product ratings relate to sales?
- Where do users drop off in the behavioral funnel?

This integration phase ensures that all downstream analysis is based on a documented and validated analytical data model.